In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM
from keras.callbacks import EarlyStopping

2025-12-06 02:42:46.372465: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-06 02:42:48.443630: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-06 02:42:52.915751: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
df_list = []

# Yelp
df_yelp = pd.read_csv('sentiment labelled sentences/yelp_labelled.txt', names=['sentence', 'label'], sep='\t')
df_yelp['source'] = 'yelp'
df_list.append(df_yelp)

# Amazon
df_amazon = pd.read_csv('sentiment labelled sentences/amazon_cells_labelled.txt', names=['sentence', 'label'], sep='\t')
df_amazon['source'] = 'amazon'
df_list.append(df_amazon)

# IMDB
df_imdb = pd.read_csv('sentiment labelled sentences/imdb_labelled.txt', names=['sentence', 'label'], sep='\t')
df_imdb['source'] = 'imdb'
df_list.append(df_imdb)

# Concatenate the dataframes
df = pd.concat(df_list)

df.head()

,sentence,label,source
0,Wow... Loved this place.,1,yelp
1,Crust is not good.,0,yelp
2,Not tasty and the texture was just nasty.,0,yelp
3,Stopped by during the late May bank holiday of...,1,yelp
4,The selection on the menu was great and so wer...,1,yelp


In [4]:
# Set the maximum number of features (words) to use
max_features = 2000
tokenizer = Tokenizer(num_words=max_features, split=' ')

# Fit the tokenizer on the sentences and convert them to sequences
tokenizer.fit_on_texts(df['sentence'].values)
X = tokenizer.texts_to_sequences(df['sentence'].values)

# Pad sequences to ensure all input vectors have the same length
X = pad_sequences(X)

# Get the labels (y)
y = df['label'].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.12)

In [7]:
def create_model():
  model = Sequential()
  # Embedding layer: max_features words, 64-dim embedding, input length is the length of the padded sequence X
  model.add(Embedding(max_features, 64, input_length=X.shape[1]))
  # LSTM layer with 16 units
  model.add(LSTM(16))
  # Dense output layer for binary classification (positive/negative)
  model.add(Dense(1, activation='sigmoid'))
  
  # Compile the model
  model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
  return model

model = create_model()

# Optional: View the model structure
# model.summary()

In [8]:
model.fit(X_train, y_train, 
          epochs=6, 
          batch_size=16, 
          validation_data=(X_test, y_test), 
          callbacks = [EarlyStopping(monitor='val_accuracy', min_delta=0.001, patience=2, verbose=1)])

Epoch 1/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 39s 244ms/step - accuracy: 0.6543 - loss: 0.6376 - val_accuracy: 0.7636 - val_loss: 0.5045
Epoch 2/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 36s 236ms/step - accuracy: 0.8495 - loss: 0.3628 - val_accuracy: 0.8091 - val_loss: 0.4814
Epoch 3/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 41s 235ms/step - accuracy: 0.9268 - loss: 0.2083 - val_accuracy: 0.8061 - val_loss: 0.4661
Epoch 4/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 41s 238ms/step - accuracy: 0.9491 - loss: 0.1417 - val_accuracy: 0.8030 - val_loss: 0.5131
Epoch 4: early stopping


In [9]:
model.save("uci_sentimentanalysis.h5")

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.DEFAULT_PROTOCOL)